In [3]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd
import requests

In [4]:
def select_first_records(csv_path, num_records):
    """
    Select the first N records from a CSV file.

    Parameters:
        csv_path (str): Path to the CSV file.
        num_records (int): Number of records to select.

    Returns:
        pandas.DataFrame: DataFrame containing the selected records.
    """
    df = pd.read_csv(csv_path)
    return df.head(num_records)


In [6]:
def make_api_request(url, params=None, headers=None, method='GET'):
    """
    Makes an HTTP request to an API.

    Parameters:
        url (str): API URL.
        params (dict, optional): Query parameters.
        headers (dict, optional): HTTP headers.
        method (str): HTTP method ('GET', 'POST', etc.).

    Returns:
        requests.Response: Response object from the request.
    """
    if headers is None:
        headers = {}
    headers['Content-Type'] = 'application/json'
    if method.upper() == 'GET':
        response = requests.get(url, params=params, headers=headers)
    elif method.upper() == 'POST':
        response = requests.post(url, json=params, headers=headers)
    else:
        raise ValueError("HTTP method not supported.")
    return response

# Trian classifier

In [7]:
json_train = {
"modelo": "st2",
"params": "separate_grid",
"embeddings_path": "./test/Embeddings",
"models_dir": "test/Modelos",
"ds_originales_path": "./data/sample.csv",
"use_adjusted": True
}

In [8]:
url_train = "http://localhost:8000/trainClassifier"
response = make_api_request(url_train, params=json_train, method='POST')
response

<Response [200]>

# Request exact

In [33]:
example = {
"modelo": "st3",
"use_adjusted": True,
"vector_input": [
    "Madrid",
    "2023",
    "Temperatura",
    "Referencia",
    "Observación"
    ]
}

In [9]:
query_ne = {'modelo': 'st2',
 'use_adjusted': True,
 'vector_input': ['Coahuila de Zaragoza.Escobedo',
  2020,
  'Total.Total',
  0.294290759,
  '-']}

In [12]:
url = "http://localhost:8000/predict"  # Cambia esto por la URL deseada
response = make_api_request(url, params=query_ne, method='POST')

In [13]:
response

<Response [500]>

In [34]:
records_exacts = select_first_records(csv_path="../data/sample.csv", num_records=18)
records_json = []
for _, row in records_exacts.iterrows():
    record = {
        "modelo": example["modelo"],
        "use_adjusted": example["use_adjusted"],
        "vector_input": row.tolist()
    }
    records_json.append(record)
records_json[0]

{'modelo': 'st3',
 'use_adjusted': True,
 'vector_input': ['Coahuila de Zaragoza.Escobedo',
  2021,
  'Total.Total',
  0.294290759,
  '-']}

In [35]:
url = "http://localhost:8000/predict"  # Cambia esto por la URL deseada
responses = []
for record in records_json:
    response = make_api_request(url, params=record, method='POST')
    responses.append(response.json())
print(responses)

[{'grupo_predicho': 10, 'modelo': 'st3', 'clasificador': 'mlp', 'confianza': 0.999998416442975, 'existe_en_grupo': True, 'indice_relativo': 0, 'indice_global': 0, 'vector_original': {'spatial': 'Coahuila de Zaragoza.Escobedo', 'temporal': 2021, 'interest': 'Total.Total', 'observation': 0.294290759, 'reference': '-'}}, {'grupo_predicho': 28, 'modelo': 'st3', 'clasificador': 'mlp', 'confianza': 0.9999954183649988, 'existe_en_grupo': True, 'indice_relativo': 0, 'indice_global': 1, 'vector_original': {'spatial': 'Sinaloa.Choix', 'temporal': 2023, 'interest': 'Mujeres.05_14', 'observation': 0.055, 'reference': '-'}}, {'grupo_predicho': 23, 'modelo': 'st3', 'clasificador': 'mlp', 'confianza': 0.99923672445343, 'existe_en_grupo': True, 'indice_relativo': 0, 'indice_global': 2, 'vector_original': {'spatial': 'San Luis Potos�.Xilitla', 'temporal': 2004, 'interest': 'Hombres.Total', 'observation': 0.385193174, 'reference': '-'}}, {'grupo_predicho': -1, 'modelo': 'st3', 'clasificador': 'mlp', 'co

In [36]:
df_responses = pd.DataFrame(responses)
df_responses.head()

,grupo_predicho,modelo,clasificador,confianza,existe_en_grupo,indice_relativo,indice_global,vector_original
0,10,st3,mlp,0.999998,True,0,0,"{'spatial': 'Coahuila de Zaragoza.Escobedo', '..."
1,28,st3,mlp,0.999995,True,0,1,"{'spatial': 'Sinaloa.Choix', 'temporal': 2023,..."
2,23,st3,mlp,0.999237,True,0,2,"{'spatial': 'San Luis Potos�.Xilitla', 'tempor..."
3,-1,st3,mlp,0.999884,True,0,3,"{'spatial': 'Oaxaca.Matías Romero Avendaño', '..."
4,34,st3,mlp,1.000000,True,0,4,"{'spatial': 'Chihuahua.Jiménez', 'temporal': 2..."


In [37]:
csv_filename = f"./df_responses_{example['modelo']}.csv"
df_responses.to_csv(csv_filename, index=False)
print(f"Archivo guardado como: {csv_filename}")

Archivo guardado como: ./df_responses_st3.csv
